In [1]:
import pandas
from Dependency.File_converter import selecting
from Dependency.one_hot_encoding import one_hot_encoding
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import GridSearchCV,HalvingGridSearchCV

data_training, data_testing = selecting()

encoded = one_hot_encoding(data_training, data_testing)
data_training = encoded["training"]
data_testing = encoded["testing"]

pandas.set_option('display.max_colwidth', None)

Forest = False
HistGB = True
SVM = False
#NaiveBayes always true to build default result

cols = ["params", "mean_test_score", "std_test_score", "rank_test_score"]

In [2]:
#Categorical_Naive_Bayes Grid

from sklearn.naive_bayes import CategoricalNB

p_grid = {
    "alpha":[0.5,1.0,2.0,5.0,10.0,15.0,20.0,25.0,30.0,40.0],
}

C_NB_grid = GridSearchCV(
    estimator=CategoricalNB(),
    param_grid=p_grid,
    scoring='accuracy',
    n_jobs=2)

def coord_positive(dataset):
    longitude = dataset["Longitude"]
    dataset["Longitude"] = longitude-dataset["Longitude"].min()
    longitude = dataset["Latitude"]
    dataset["Latitude"] = longitude-dataset["Latitude"].min()
    return dataset

data_training_positive = coord_positive(data_training)

data_training_X = data_training_positive.loc[ : , data_training_positive.columns != "Accident_Severity"]
data_training_Y = data_training_positive["Accident_Severity"]

C_NB_grid.fit(data_training_X, data_training_Y)

results = pandas.DataFrame(C_NB_grid.cv_results_)
    
results[cols].sort_values("rank_test_score").head(10)

,params,mean_test_score,std_test_score,rank_test_score
9,{'alpha': 40.0},0.493274,0.006666,1
6,{'alpha': 20.0},0.493067,0.006204,2
5,{'alpha': 15.0},0.493067,0.005846,2
7,{'alpha': 25.0},0.493037,0.006233,4
8,{'alpha': 30.0},0.492889,0.006344,5
3,{'alpha': 5.0},0.492770,0.006759,6
4,{'alpha': 10.0},0.492533,0.005816,7
2,{'alpha': 2.0},0.491852,0.007004,8
1,{'alpha': 1.0},0.490993,0.006541,9
0,{'alpha': 0.5},0.489422,0.006317,10


In [3]:
#Random_Forest Grid

from sklearn.ensemble import RandomForestClassifier

p_grid = {
    "n_estimators": [80,100,120,140],
    "criterion": ["gini", "entropy", "log_loss"],
    "min_samples_split": [2,20,100],
    "min_samples_leaf": [1,5,20],
    "min_weight_fraction_leaf": [0.0,0.0001,0.001,0.01],
    "max_features": ["sqrt","log2"],
    "min_impurity_decrease": [0.0,0.0001,0.001,0.01],
    "ccp_alpha": [0.0,0.1,1.0],
}

Forest_grid = HalvingGridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=p_grid,
    scoring='accuracy',
    max_resources=500,
    n_jobs=2)

data_training_X = data_training.loc[ : , data_training.columns != "Accident_Severity"]
data_training_Y = data_training["Accident_Severity"]

if(Forest):
    Forest_grid.fit(data_training_X,data_training_Y)
    results = pandas.DataFrame(Forest_grid.cv_results_)

# Compact view of param, 10 worst
results[cols].sort_values("rank_test_score").tail(10)

,params,mean_test_score,std_test_score,rank_test_score
9,{'alpha': 40.0},0.493274,0.006666,1
6,{'alpha': 20.0},0.493067,0.006204,2
5,{'alpha': 15.0},0.493067,0.005846,2
7,{'alpha': 25.0},0.493037,0.006233,4
8,{'alpha': 30.0},0.492889,0.006344,5
3,{'alpha': 5.0},0.492770,0.006759,6
4,{'alpha': 10.0},0.492533,0.005816,7
2,{'alpha': 2.0},0.491852,0.007004,8
1,{'alpha': 1.0},0.490993,0.006541,9
0,{'alpha': 0.5},0.489422,0.006317,10


In [4]:
# Compact view of param, 10 best
results[cols].sort_values("rank_test_score").head(10)

,params,mean_test_score,std_test_score,rank_test_score
9,{'alpha': 40.0},0.493274,0.006666,1
6,{'alpha': 20.0},0.493067,0.006204,2
5,{'alpha': 15.0},0.493067,0.005846,2
7,{'alpha': 25.0},0.493037,0.006233,4
8,{'alpha': 30.0},0.492889,0.006344,5
3,{'alpha': 5.0},0.492770,0.006759,6
4,{'alpha': 10.0},0.492533,0.005816,7
2,{'alpha': 2.0},0.491852,0.007004,8
1,{'alpha': 1.0},0.490993,0.006541,9
0,{'alpha': 0.5},0.489422,0.006317,10


In [ ]:
#Hist_Grabient_Boosting Grid

from sklearn.ensemble import HistGradientBoostingClassifier

p_grid = {
    "learning_rate":[0.03,0.04],
    "max_iter":[180],
    "max_leaf_nodes":[31],
    "min_samples_leaf":[30],
    "max_features":[0.4,0.5,0.6],
    "categorical_features":[["Urban_or_Rural_Area","Did_Police_Officer_Attend_Scene_of_Accident"]],
    "scoring":['accuracy'],
    "early_stopping":[False],
    "validation_fraction":[0.001],
}

#HalvingGridSearchCV / GridSearchCV
H_GB_grid = GridSearchCV(
    estimator=HistGradientBoostingClassifier(),
    param_grid=p_grid,
    scoring='accuracy',
    #max_resources=1000,
    n_jobs=2)

data_training_X = data_training.loc[ : , data_training.columns != "Accident_Severity"]
data_training_Y = data_training["Accident_Severity"]

if(HistGB):
    H_GB_grid.fit(data_training_X,data_training_Y)
    results = pandas.DataFrame(H_GB_grid.cv_results_)

# Compact view of param, 10 worst
results[cols].sort_values("rank_test_score").tail(10)


In [ ]:
# Compact view of param, 10 best
results[cols].sort_values("rank_test_score").head(10)

,params,mean_test_score,std_test_score,rank_test_score
4,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.04, 'max_features': 0.5, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.520385,0.004174,1
3,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.03, 'max_features': 0.6, 'max_iter': 200, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.520356,0.004685,2
1,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.03, 'max_features': 0.5, 'max_iter': 200, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.519852,0.004498,3
6,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.04, 'max_features': 0.6, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.519378,0.005545,4
0,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.03, 'max_features': 0.5, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.519052,0.004664,5
2,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.03, 'max_features': 0.6, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.518785,0.005094,6
8,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.05, 'max_features': 0.5, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.518667,0.002602,7
11,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.05, 'max_features': 0.6, 'max_iter': 200, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 's

In [ ]:
#SVM_(SVC) Grid

from sklearn.svm import SVC

p_grid = {
    "C":[1.0], 
    "kernel":['rbf'], 
    "degree":[3], 
    "gamma":['scale'], 
    "coef0":[0.0], 
    "shrinking":[True], 
    "probability":['deprecated'], 
    "max_iter":[-1], 
    "decision_function_shape":['ovr'], 
    "break_ties":[False], 
}

SVM_grid = HalvingGridSearchCV(
    estimator=SVC(),
    param_grid=p_grid,
    scoring='accuracy',
    max_resources=500,
    n_jobs=2)

data_training_X = data_training.loc[ : , data_training.columns != "Accident_Severity"]
data_training_Y = data_training["Accident_Severity"]

if(SVM):
    SVM_grid.fit(data_training_X,data_training_Y)
    results = pandas.DataFrame(SVM_grid.cv_results_)

# Compact view of param, 10 worst
results[cols].sort_values("rank_test_score").tail(10)

,params,mean_test_score,std_test_score,rank_test_score
1,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.03, 'max_features': 0.5, 'max_iter': 200, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.519852,0.004498,3
6,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.04, 'max_features': 0.6, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.519378,0.005545,4
0,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.03, 'max_features': 0.5, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.519052,0.004664,5
2,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.03, 'max_features': 0.6, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.518785,0.005094,6
8,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.05, 'max_features': 0.5, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.518667,0.002602,7
11,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.05, 'max_features': 0.6, 'max_iter': 200, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.518193,0.005030,8
5,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.04, 'max_features': 0.5, 'max_iter': 200, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.517985,0.004038,9
7,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.04, 'max_features': 0.6, 'max_iter': 200, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 's

In [ ]:
# Compact view of param, 10 best
results[cols].sort_values("rank_test_score").head(10)

,params,mean_test_score,std_test_score,rank_test_score
4,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.04, 'max_features': 0.5, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.520385,0.004174,1
3,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.03, 'max_features': 0.6, 'max_iter': 200, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.520356,0.004685,2
1,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.03, 'max_features': 0.5, 'max_iter': 200, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.519852,0.004498,3
6,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.04, 'max_features': 0.6, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.519378,0.005545,4
0,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.03, 'max_features': 0.5, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.519052,0.004664,5
2,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.03, 'max_features': 0.6, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.518785,0.005094,6
8,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.05, 'max_features': 0.5, 'max_iter': 180, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 'scoring': 'accuracy', 'validation_fraction': 0.001}",0.518667,0.002602,7
11,"{'categorical_features': ['Day_of_Week', '1st_Road_Class', 'Road_Type', 'Junction_Control', '2nd_Road_Class', 'Pedestrian_Crossing-Physical_Facilities', 'Light_Conditions', 'Weather_Conditions', 'Road_Surface_Conditions', 'Urban_or_Rural_Area', 'Did_Police_Officer_Attend_Scene_of_Accident'], 'early_stopping': False, 'learning_rate': 0.05, 'max_features': 0.6, 'max_iter': 200, 'max_leaf_nodes': 31, 'min_samples_leaf': 30, 's